##### ARTI 560 - Computer Vision

## Action Recognition - Exercise

### Objective

In this exercise, you will train a deep learning model to recognize three specific human actions using the [UCF11 (YouTube Action) dataset](https://www.crcv.ucf.edu/data/UCF_YouTube_Action.php) and validate the model's real-world performance using external video data.

*[Note: This notebook is based on [this](https://github.com/Sumaya2026/learnopencv/tree/master/Optical-Flow-Estimation-using-Deep-Learning-RAFT) GitHub Repository by LearnOpenCV]*


#### Tasks

- Choose **three classes** from the UCF11 dataset (e.g., Basketball Shooting, Biking, Tennis Swinging, etc.).
- Preprocess the dataset.
- Split the data into training and testing.
- Create and train the model.
- Save the trained model .
    **Important Note**: The final trained model must be saved with a filename that includes your name. This is a mandatory step for the submission.
    ```
    # Example Code
    student_name = "YourName" # Replace with your actual name
    save_path = f"{student_name}_ucf11_model.h5"
    model.save(save_path)
    print(f"Model saved as {save_path}")
    ```
- Validate the model on 3 Youtube videos, each clearly showing one of your three chosen action classes.


In [1]:
import os
import cv2
import numpy as np
import random
import tensorflow as tf

from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import ConvLSTM2D, MaxPooling3D, TimeDistributed
from tensorflow.keras.layers import Dropout, Flatten, Dense

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!ls /content/drive/MyDrive

'0. Intro-to-Java-Programming-master'
 1.jpg
 2.jpg
 3.jpg
 4.jpg
 5.jpg
'about the graph can you give me the excel sheet.gsheet'
 cleaned_sharqiyah_rentals.csv
'Colab Notebooks'
'Copy of records.zip'
 CS211_assignment2.docx
 encoder_columns.pkl
 FER-2013.zip
 final_model.pkl
'for 3,4 and conclusion can you make it shorter.gsheet'
'General letter COOP 2026- IAU copy.pdf'
'Java Ex'
 LinearAlgebra
'Mask R-CNN'
 mask-rcnn-inception-resnet-v2-tensorflow2-1024x1024-v1
 Mydrive
'Ohoud Saleh 2.pdf'
'Ohoud Saleh Alqrai 2.pdf'
'Ohoud Saleh Alqrai.pdf'
'Ohoud Saleh.pdf'
 OhoudSaleh.pdf
'Transcript 2.pdf'
'Tutorial Chapter 3'
 untitled
'Untitled (1).pdf'
 xgboost_model.pkl
 X_test.csv
 X_train.csv
 y_test.csv
 y_train.csv
 ‎⁨تقويم_سنَد_للفصل_الدراسي_الاول_لعام_١٤٤٧هـ⁩.pdf
'ريادة الاعمال'
'محطات التحلية.dwg'
'نموذج بدون عنوان (1).gform'
'نموذج بدون عنوان.gform'


In [4]:
IMAGE_HEIGHT, IMAGE_WIDTH = 64, 64
SEQUENCE_LENGTH = 20
DATASET_DIR = "/content/drive/MyDrive/Mydrive/UCF11_updated_mpg" 
CLASSES_LIST = ["basketball", "biking", "tennis_swing"]

In [5]:
def frames_extraction(video_path):
    frames_list = []
    video_reader = cv2.VideoCapture(video_path)

    video_frames_count = int(video_reader.get(cv2.CAP_PROP_FRAME_COUNT))
    skip_frames_window = max(int(video_frames_count / SEQUENCE_LENGTH), 1)

    for frame_counter in range(SEQUENCE_LENGTH):
        video_reader.set(cv2.CAP_PROP_POS_FRAMES, frame_counter * skip_frames_window)
        success, frame = video_reader.read()

        if not success:
            break

        resized_frame = cv2.resize(frame, (IMAGE_WIDTH, IMAGE_HEIGHT))
        normalized_frame = resized_frame / 255.0
        frames_list.append(normalized_frame)

    video_reader.release()

    return frames_list

In [6]:
def create_dataset():
    features = []
    labels = []

    for class_index, class_name in enumerate(CLASSES_LIST):
        print(f"Extracting data for class: {class_name}")
        class_folder_path = os.path.join(DATASET_DIR, class_name)

        for root, dirs, files in os.walk(class_folder_path):
            for file_name in files:
                video_path = os.path.join(root, file_name)

                frames = frames_extraction(video_path)

                if len(frames) == SEQUENCE_LENGTH:
                    features.append(frames)
                    labels.append(class_index)

    return np.asarray(features), np.array(labels)

In [7]:
features, labels = create_dataset()
print(features.shape)
print(labels.shape)

Extracting data for class: basketball
Extracting data for class: biking
Extracting data for class: tennis_swing
(450, 20, 64, 64, 3)
(450,)


In [8]:
one_hot_encoded_labels = to_categorical(labels, num_classes=len(CLASSES_LIST))

In [9]:
print("DATASET_DIR =", DATASET_DIR)
!find "$DATASET_DIR" -maxdepth 5 -type f | head -30

DATASET_DIR = /content/drive/MyDrive/Mydrive/UCF11_updated_mpg
/content/drive/MyDrive/Mydrive/UCF11_updated_mpg/.DS_Store
/content/drive/MyDrive/Mydrive/UCF11_updated_mpg/biking/v_biking_05/v_biking_05_03.mpg
/content/drive/MyDrive/Mydrive/UCF11_updated_mpg/biking/v_biking_05/v_biking_05_09.mpg
/content/drive/MyDrive/Mydrive/UCF11_updated_mpg/biking/v_biking_05/v_biking_05_08.mpg
/content/drive/MyDrive/Mydrive/UCF11_updated_mpg/biking/v_biking_05/v_biking_05_07.mpg
/content/drive/MyDrive/Mydrive/UCF11_updated_mpg/biking/v_biking_05/v_biking_05_05.mpg
/content/drive/MyDrive/Mydrive/UCF11_updated_mpg/biking/v_biking_05/v_biking_05_04.mpg
/content/drive/MyDrive/Mydrive/UCF11_updated_mpg/biking/v_biking_05/v_biking_05_02.mpg
/content/drive/MyDrive/Mydrive/UCF11_updated_mpg/biking/v_biking_05/v_biking_05_01.mpg
/content/drive/MyDrive/Mydrive/UCF11_updated_mpg/biking/v_biking_05/v_biking_05_06.mpg
/content/drive/MyDrive/Mydrive/UCF11_updated_mpg/biking/v_biking_03/v_biking_03_01.mpg
/content

In [10]:
features_train, features_test, labels_train, labels_test = train_test_split(
    features,
    one_hot_encoded_labels,
    test_size=0.25,
    shuffle=True,
    random_state=42
)

In [11]:
model = Sequential()

model.add(ConvLSTM2D(
    filters=4,
    kernel_size=(3, 3),
    activation='tanh',
    data_format="channels_last",
    recurrent_dropout=0.2,
    return_sequences=True,
    input_shape=(SEQUENCE_LENGTH, IMAGE_HEIGHT, IMAGE_WIDTH, 3)
))

model.add(MaxPooling3D(pool_size=(1, 2, 2), padding='same'))
model.add(TimeDistributed(Dropout(0.2)))

model.add(ConvLSTM2D(
    filters=8,
    kernel_size=(3, 3),
    activation='tanh',
    recurrent_dropout=0.2,
    return_sequences=True
))

model.add(MaxPooling3D(pool_size=(1, 2, 2), padding='same'))
model.add(TimeDistributed(Dropout(0.2)))

model.add(Flatten())
model.add(Dense(len(CLASSES_LIST), activation="softmax"))

model.summary()

model.compile(
    loss="categorical_crossentropy",
    optimizer="Adam",
    metrics=["accuracy"]
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv_lstm2d (ConvLSTM2D)        │ (None, 20, 62, 62, 4)  │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d (MaxPooling3D)    │ (None, 20, 31, 31, 4)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 20, 31, 31, 4)  │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_lstm2d_1 (ConvLSTM2D)      │ (None, 20, 29, 29, 8)  │         3,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_1 (MaxPooling3D)  │ (None, 20, 15, 15, 8)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 20, 15, 15, 8)  │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │       108,003 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 112,515 (439.51 KB)

 Trainable params: 112,515 (439.51 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint_path = "/content/drive/MyDrive/Mydrive/ucf11_checkpoint.keras"

checkpoint = ModelCheckpoint(
    checkpoint_path,
    save_best_only=False,
    save_weights_only=False
)

history = model.fit(
    features_train,
    labels_train,
    epochs=20,
    batch_size=4,
    validation_split=0.2,
    shuffle=True,
    callbacks=[checkpoint]
)

Epoch 1/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 243s 3s/step - accuracy: 0.4758 - loss: 1.2357 - val_accuracy: 0.3382 - val_loss: 1.4127
Epoch 2/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 201s 3s/step - accuracy: 0.6208 - loss: 0.8738 - val_accuracy: 0.3529 - val_loss: 1.6396
Epoch 3/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 198s 3s/step - accuracy: 0.7398 - loss: 0.6851 - val_accuracy: 0.4265 - val_loss: 1.5868
Epoch 4/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 121s 2s/step - accuracy: 0.7918 - loss: 0.5329 - val_accuracy: 0.4265 - val_loss: 2.2530
Epoch 5/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 110s 2s/step - accuracy: 0.8625 - loss: 0.4060 - val_accuracy: 0.5735 - val_loss: 1.3413
Epoch 6/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 107s 2s/step - accuracy: 0.9071 - loss: 0.2950 - val_accuracy: 0.6176 - val_loss: 1.5502
Epoch 7/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 107s 2s/step - accuracy: 0.8885 - loss: 0.3444 - val_accuracy: 0.7353 - val_loss: 0.6439
Epoch 8/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 142s 2s/step - accuracy: 0.9033 - loss: 0.2455 - val_accuracy: 0.7353 - v

In [15]:
#from google.colab import drive
#drive.mount('/content/drive')

student_name = "OhoudSaleh"
save_path = f"/content/drive/MyDrive/Mydrive/{student_name}_ucf11_model.h5"

model.save(save_path)
print(f"Model saved as {save_path}")

Model saved as /content/drive/MyDrive/Mydrive/OhoudSaleh_ucf11_model.h5


In [17]:
def predict_action(video_path):
    frames = frames_extraction(video_path)

    if len(frames) != SEQUENCE_LENGTH:
        print("Not enough frames in video.")
        return

    frames = np.expand_dims(frames, axis=0)

    predictions = model.predict(frames)[0]
    predicted_class_index = np.argmax(predictions)
    predicted_class_name = CLASSES_LIST[predicted_class_index]
    confidence = predictions[predicted_class_index]

    print("Predicted Action:", predicted_class_name)
    print("Confidence:", confidence)

In [ ]:
basketball_url = "https://www.youtube.com/shorts/qsWvFwBwzQ4"
biking_url = "https://www.youtube.com/watch?v=m52SZkVAiSA"
tennis_url = "https://www.youtube.com/watch?v=yyQ-v4V3NU8"

In [18]:
!pip install yt-dlp -q

!yt-dlp -f mp4 -o basketball.mp4 "https://www.youtube.com/shorts/qsWvFwBwzQ4"
!yt-dlp -f mp4 -o biking.mp4 "https://www.youtube.com/watch?v=m52SZkVAiSA"
!yt-dlp -f mp4 -o tennis.mp4 "https://www.youtube.com/watch?v=yyQ-v4V3NU8"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 33.8 MB/s eta 0:00:0000:0100:01
         Pre-merged mp4 formats are not available from all sites, or may only be available in lower quality.
         To prioritize the best h264 video and aac audio in an mp4 container, use "-t mp4" instead.
         If you know what you are doing and want a pre-merged mp4 format, use "-f b[ext=mp4]" instead to suppress this warning
[youtube] Extracting URL: https://www.youtube.com/shorts/qsWvFwBwzQ4
[youtube] qsWvFwBwzQ4: Downloading webpage
[youtube] qsWvFwBwzQ4: Downloading android vr player API JSON
[info] qsWvFwBwzQ4: Downloading 1 format(s): 18
[download] Destination: basketball.mp4
[download] 100% of    8.88MiB in 00:00:00 at 12.58MiB/s;33m00:000m
         Pre-merged mp4 formats are not available from all sites, or may only be available in lower quality.
         To prioritize the best h264 video and aac audio in

In [19]:
predict_action("basketball.mp4")
predict_action("biking.mp4")
predict_action("tennis.mp4")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Predicted Action: biking
Confidence: 0.9994709
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step
Predicted Action: biking
Confidence: 0.9993722
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step
Predicted Action: biking
Confidence: 0.99996686


In [20]:
import numpy as np

unique, counts = np.unique(labels, return_counts=True)
print(dict(zip(unique, counts)))

{np.int64(0): np.int64(138), np.int64(1): np.int64(145), np.int64(2): np.int64(167)}


In [21]:
for i in range(10):
    pred = model.predict(np.expand_dims(features_test[i], axis=0))[0]
    print("Predicted:", CLASSES_LIST[np.argmax(pred)])
    print("Actual:", CLASSES_LIST[np.argmax(labels_test[i])])
    print(pred)
    print()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step
Predicted: tennis_swing
Actual: tennis_swing
[4.4862190e-05 2.4114036e-13 9.9995518e-01]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step
Predicted: tennis_swing
Actual: tennis_swing
[3.4832513e-01 1.1820437e-05 6.5166301e-01]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step
Predicted: basketball
Actual: basketball
[9.9998569e-01 1.9859131e-06 1.2259097e-05]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 172ms/step
Predicted: tennis_swing
Actual: basketball
[0.33610547 0.08952483 0.5743697 ]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 177ms/step
Predicted: tennis_swing
Actual: tennis_swing
[9.4063056e-05 2.3955151e-03 9.9751043e-01]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step
Predicted: biking
Actual: biking
[4.085662e-08 1.000000e+00 4.147367e-08]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step
Predicted: tennis_swing
Actual: tennis_swing
[3.1237088e-12 1.4794552e-05 9.9998522e-01]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step
Predicted: basketball
Actual: tennis_swing
[5.7771212e-01 1.1170994e-06 4.2228669e-01]



In [ ]:
basketball_path = "/content/drive/MyDrive/Mydrive/basketball.mp4"
biking_path = "/content/drive/MyDrive/Mydrive/biking.mp4"
tennis_path = "/content/drive/MyDrive/Mydrive/tennis.mp4"

In [ ]:
predict_action(basketball_path)
predict_action(biking_path)
predict_action(tennis_path)